In [ ]:
# %% [CELL 1] Configuration  <- the only cell you edit
# =============================================================================
# L8 PROBE — does exact, weighted token search recover the blocking misses?
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# Builds, per country, over the FULL train pool:
#   L8   exact typed-token search, weighted by rarity (IDF) and by token type:
#        n  name words             nb name word pairs      nf name without spaces
#        k  skeleton words         ks sorted-letter keys (swapped letters)
#        a  address words          d  numbers              ap address word pairs
#        Non-Latin records use their IndicTrans2 translation for name tokens,
#        plus skeleton words from both the translation and the transliteration.
#        Tokens found in more than DF_CAP records are ignored (noise + speed).
#   L2t  name+address TF-IDF search over ONLY the non-Latin records, with their
#        translated names (GPU; skipped if no GPU).
# Queries: a random sample of held-out entities + every S1 entity with a known
# miss in blocking_misses.csv (capped per country).
# Reports: recall gain over old + Phase B, recovery of known misses by
# category, three token-weight settings, speed and projected full-run time.
#
# NEW NOTEBOOK. Inputs: (1) competition dataset, (2) ckpt_train zips dataset,
# (3) Phase B TRAIN notebook output, (4) translation pre-step TRAIN output
# (translations_train.parquet), (5) blocking_misses.csv.
# SETTINGS GPU T4 x1 (for L2t; the rest is CPU), Internet ON, Persistence Files.
# RUNTIME  ~25-40 min.
# =============================================================================

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

N_HELDOUT = 5_000          # random held-out queries per country (hash buckets < 500)
N_MISS_Q = 15_000          # max S1 entities with known misses per country
K8, K2T = 100, 30          # neighbours kept per query: L8, L2t
DF_CAP = 2_000             # ignore tokens present in more records than this
MAX_ADDR_TOKENS = 8        # address word pairs are built from the first 8 address tokens
# Each field is scored on its own (cosine over its token types, IDF-weighted), then
# score = (w_name * name_sim + w_addr * addr_sim) / (weights of the fields BOTH
# records have). A blank-address record is judged on its name alone, instead of
# being outweighed by the ~36 address tokens of the query.
FIELDS = {"name": ["n", "nb", "nf", "k", "ks"], "addr": ["a", "d", "ap"]}
TYPE_W = {"n": 1.0, "nb": 1.0, "nf": 1.0, "k": 1.0, "ks": 0.7, "a": 1.0, "d": 1.0, "ap": 1.0}
WEIGHTS = {                # field weights tried by the probe
    "balanced":      {"name": 1.0, "addr": 1.0},
    "address-heavy": {"name": 1.0, "addr": 1.5},
    "name-heavy":    {"name": 1.5, "addr": 1.0},
}
Q_CHUNK = 2_000            # queries per sparse product
TOK_CHUNK = 100_000        # records per token-building task (parallel)
DIM_L2T, MIN_SIM = 256, 0.10
TFIDF_FIT_ROWS, SVD_FIT_ROWS, TFIDF_CHUNK = 1_000_000, 300_000, 250_000
Q_BATCH = 1024
SEED = 42

In [ ]:
# %% [CELL 2] Imports and helpers
# ---------------------------------------------------------------------------
# Logging, timing, zip-aware input finder (unpacks attached zips once).
# ---------------------------------------------------------------------------
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm

try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]


def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)


def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)


def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)


def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0


def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000


def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits


def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total

In [ ]:
# %% [CELL 3] Text cleaning + transliteration (shared with the pipeline)
# ---------------------------------------------------------------------------
# Same cleaning as every other stage.
# ---------------------------------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]


def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)


# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")

ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]


def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}


def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))


def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out




# ---- v2: transliteration, legal form, phonetic skeleton (shared) -------------
import re as _re
import unicodedata as _ud
try:
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
except ImportError:
    print("installing indic_transliteration (MIT licence; needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic_transliteration"], check=True)
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit

_BLOCK_SCRIPT = {0x0900: _sanscript.DEVANAGARI, 0x0980: _sanscript.BENGALI,
                 0x0A00: _sanscript.GURMUKHI, 0x0A80: _sanscript.GUJARATI,
                 0x0B00: _sanscript.ORIYA, 0x0B80: _sanscript.TAMIL,
                 0x0C00: _sanscript.TELUGU, 0x0C80: _sanscript.KANNADA,
                 0x0D00: _sanscript.MALAYALAM}
_LEGAL_TOKENS = {"pvt", "private", "pr", "pra", "ltd", "limited", "li", "llp", "pllc", "llc",
                 "lp", "inc", "incorporated", "corp", "corporation", "co", "company", "sas",
                 "sasu", "sarl", "eurl", "snc", "sa", "public"}
_LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")    # transliterated private / limited / LLP
_FORM_KEYS = (("LLP", {"llp"}), ("PLLC", {"pllc"}), ("LLC", {"llc"}), ("LP", {"lp"}),
              ("INC", {"inc", "incorporated"}), ("CORP", {"corp", "corporation"}),
              ("SASU", {"sasu"}), ("SAS", {"sas"}), ("SARL", {"sarl"}), ("EURL", {"eurl"}),
              ("SNC", {"snc"}), ("SA", {"sa"}))
_SKEL_FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
              ("dh", "d"), ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"),
              ("z", "s"), ("w", "v"), ("j", "g"), ("y", ""), ("v", "b"))   # Bengali renders B as v


def latin_tokens(name):
    """Lowercase Latin tokens of a name. Indic-script words are transliterated
    (ITRANS), accents removed, dots dropped (L.L.C -> llc), other punctuation
    splits words (Housing-Network -> housing network)."""
    s = str(name)
    s = _re.sub(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", " ", s, flags=_re.I)
    s = s.replace(".", "")
    fast = all(ord(ch) <= 0x024F for ch in s)
    if fast:
        s = _re.sub(r"[^\w\s]", " ", s)
    else:
        # Indic vowel signs / virama are not \w, so only replace real punctuation
        # and symbols (Unicode categories P* and S*).
        s = "".join(" " if _ud.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        indic = False
        if not fast:
            blk = next((b for b in _BLOCK_SCRIPT if any(b <= ord(c) < b + 0x80 for c in w)), None)
            if blk is not None:
                indic = True
                try:
                    w = _translit(w, _BLOCK_SCRIPT[blk], _sanscript.ITRANS)
                except Exception:
                    pass
                if blk == 0x0B80:                     # Tamil: library leaves some letters
                    w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = _ud.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if indic and blk == 0x0B80:
            # Tamil has no aspirated/voiced contrast; the library writes ச க ப ட/த
            # as jh gh bh dh — fold them to s k p t.
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = _re.sub(r"[^a-z0-9]", "", w)
        if indic:
            w = _re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)    # inherent final 'a'
        if w:
            out.append(w)
    return out


def legal_form(toks):
    """Canonical legal form: PVT (private limited), PLC, LTD, LLP, LLC, INC, ...
    Spelling variants map to one code (Pvt Ltd = Private Limited = प्रा. लि.)."""
    s = set(toks)
    has_pvt = bool(s & {"pvt", "private", "pr", "pra"}) or any(t.startswith(("prai", "pirai")) for t in toks)
    has_ltd = bool(s & {"ltd", "limited", "li"}) or any(t.startswith("limi") for t in toks)
    if has_pvt:
        return "PVT"
    if "public" in s and has_ltd:
        return "PLC"
    if any(t.startswith("elel") for t in toks):
        return "LLP"
    for code, keys in _FORM_KEYS:
        if s & keys:
            return code
    if has_ltd:
        return "LTD"
    if s & {"co", "company"}:
        return "CO"
    return ""

In [ ]:
# %% [CELL 4] L8 token builder + GPU helpers for L2t
# ---------------------------------------------------------------------------
# Typed tokens with exact ids, parallel build; GPU top-k and TF-IDF for L2t.
# ---------------------------------------------------------------------------
# ---- L8 token builder ---------------------------------------------------------
# Tokens get EXACT column ids: each token string -> 64-bit fingerprint (Python
# hash; identical in every forked worker of this session) -> numbered exactly
# with pandas. No shared hash buckets, so rare tokens stay rare.
import os
import multiprocessing as _mp
import scipy.sparse as sp
from concurrent.futures import ProcessPoolExecutor

TYPES = ["n", "nb", "nf", "k", "ks", "a", "d", "ap"]
_FOLD = tuple(_SKEL_FOLD)


def skel_word(w):
    """Phonetic skeleton of one word (m->n BEFORE collapsing repeats: fixes the
    Phase B order bug, so 'investments' -> 'nvstnts')."""
    for a, b in _FOLD:
        w = w.replace(a, b)
    w = _re.sub(r"[aeiou]", "", w).replace("m", "n")
    return _re.sub(r"(.)\1+", r"\1", w)


def _is_legal(w):
    return w in _LEGAL_TOKENS or w.startswith(_LEGAL_PREFIX)


def record_tokens(name_words, raw_name, translation, caddr):
    """Typed tokens of one record (sets, so repeats count once)."""
    T = {t: set() for t in TYPES}
    words = [w for w in name_words if len(w) >= 2]
    T["n"].update(words)
    T["nb"].update(f"{a}_{b}" for a, b in zip(words, words[1:]))
    full = "".join(words)
    if len(full) >= 4:
        T["nf"].add(full)
    src = [w for w in latin_tokens(raw_name) if not _is_legal(w)] + words
    if translation:
        src += [w for w in latin_tokens(translation) if not _is_legal(w)]
    for w in src:
        s = skel_word(w)
        if len(s) >= 2:
            T["k"].add(s)
        if len(s) >= 3:
            T["ks"].add("".join(sorted(s)))
    atoks = caddr.split()
    for w in atoks:
        if w.isdigit():
            if w.lstrip("0"):
                T["d"].add(w.lstrip("0"))
        elif len(w) >= 2:
            T["a"].add(w)
    firsts = list(dict.fromkeys(w.lstrip("0") or w if w.isdigit() else w for w in atoks))[:MAX_ADDR_TOKENS]
    for i in range(len(firsts)):
        for j in range(i + 1, len(firsts)):
            a, b = sorted((firsts[i], firsts[j]))
            T["ap"].add(f"{a}|{b}")
    return T


def _token_task(args):
    """Worker: records -> per token type (tokens per record, flat fingerprints)."""
    names, addrs, trans = args
    base = pd.Series([t if t else n for n, t in zip(names, trans)])
    nw = clean_name(base).str.split().tolist()
    ca = clean_addr(pd.Series(addrs)).tolist()
    per = {t: [] for t in TYPES}
    for w, n, tr, a in zip(nw, names, trans, ca):
        T = record_tokens(w, n, tr, a)
        for t in TYPES:
            per[t].append(T[t])
    out = {}
    for t in TYPES:
        lens = np.fromiter((len(x) for x in per[t]), dtype=np.int32, count=len(per[t]))
        flat = np.fromiter((hash(x) for s in per[t] for x in s), dtype=np.int64, count=int(lens.sum()))
        out[t] = (lens, flat)
    return out


def build_tokens(names, addrs, trans, desc):
    """Typed token fingerprints for many records, in parallel over CPU cores."""
    tasks = [(names[i:i + TOK_CHUNK], addrs[i:i + TOK_CHUNK], trans[i:i + TOK_CHUNK])
             for i in range(0, len(names), TOK_CHUNK)]
    parts = []
    ctx = _mp.get_context("fork")            # same hash seed in every worker
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2, mp_context=ctx) as ex:
        for r in bar(ex.map(_token_task, tasks), total=len(tasks), desc=desc, unit="chunk"):
            parts.append(r)
    return {t: (np.concatenate([p[t][0] for p in parts]), np.concatenate([p[t][1] for p in parts]))
            for t in TYPES}


def to_csr(lens, flat, vocab):
    """Binary record x token matrix; tokens missing from `vocab` are dropped."""
    codes = vocab.get_indexer(flat)
    rows = np.repeat(np.arange(len(lens), dtype=np.int64), lens)
    keep = codes >= 0
    X = sp.csr_matrix((np.ones(int(keep.sum()), np.float32), (rows[keep], codes[keep])),
                      shape=(len(lens), len(vocab)))
    X.sum_duplicates()
    X.data[:] = 1.0
    return X


import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize


def purge():
    gc.collect()
    torch.cuda.empty_cache()


# ---- GPU exact top-k (same algorithm as Notebook 2) ---------------------------
def gpu_topk(Q, P, k, desc, q_batch=Q_BATCH, p_chunk=1_000_000):
    """Exact inner-product top-k on the GPU (fp16); -1 where score < MIN_SIM."""
    k = min(k, P.shape[0])
    P_t = torch.from_numpy(np.ascontiguousarray(P, dtype=np.float16)).to(DEV)
    I = np.empty((Q.shape[0], k), dtype=np.int32)
    S = np.empty((Q.shape[0], k), dtype=np.float16)
    for i in bar(range(0, Q.shape[0], q_batch), desc=desc, unit="batch"):
        q = torch.from_numpy(np.ascontiguousarray(Q[i:i + q_batch], dtype=np.float16)).to(DEV)
        best_v = best_i = None
        for j in range(0, P_t.shape[0], p_chunk):
            s = q @ P_t[j:j + p_chunk].T
            v, ix = torch.topk(s, min(k, s.shape[1]), dim=1)
            ix += j
            if best_v is None:
                best_v, best_i = v, ix
            else:
                cv, ci = torch.cat([best_v, v], dim=1), torch.cat([best_i, ix], dim=1)
                best_v, sel = torch.topk(cv, k, dim=1)
                best_i = torch.gather(ci, 1, sel)
            del s
        n = q.shape[0]
        I[i:i + n] = best_i.cpu().numpy()
        S[i:i + n] = best_v.cpu().numpy()
    del P_t
    purge()
    I[S < MIN_SIM] = -1
    return I, S


def fit_tfidf_svd(p_txt, dim, ngram):
    """Char n-gram TF-IDF (vocabulary from a sample) + SVD fitted on a sample."""
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(p_txt), min(TFIDF_FIT_ROWS, len(p_txt)), replace=False)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, min_df=3, max_df=0.5,
                          max_features=40_000, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform([p_txt[i] for i in idx])
    dim = min(dim, X.shape[1] - 1)
    sidx = rng.choice(X.shape[0], min(SVD_FIT_ROWS, X.shape[0]), replace=False)
    svd = TruncatedSVD(n_components=dim, algorithm="randomized", n_iter=4,
                       random_state=SEED).fit(X[sidx])
    return vec, svd


def tfidf_transform(vec, svd, texts, desc):
    out = np.empty((len(texts), svd.n_components), dtype=np.float16)
    for i in bar(range(0, len(texts), TFIDF_CHUNK), desc=desc, unit="chunk"):
        out[i:i + TFIDF_CHUNK] = normalize(svd.transform(vec.transform(texts[i:i + TFIDF_CHUNK])))
    return out

In [ ]:
# %% [CELL 5] Inputs
import torch
HAS_GPU = torch.cuda.is_available()
DEV = "cuda" if HAS_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (L2t will be skipped)")

OLD = {f.parent.name: f.parent for f in locate("pairs.npz")
       if f.parent.parent.name == "ckpt_train" and (f.parent / "qids.npy").exists()}
NEWB = {f.parent.name: f.parent for f in locate("pairs_b.npz") if f.parent.parent.name == "ckpt_b_train"}
assert OLD, "ckpt_train not found: attach the ckpt_train zips dataset"
assert NEWB, "ckpt_b_train not found: attach the Phase B TRAIN notebook output"
tr_files = sorted(INPUT_ROOT.rglob("translations_train.parquet")) + sorted(WORK.glob("translations_train.parquet"))
TRANS = {}
if tr_files:
    t = pd.read_parquet(tr_files[0], columns=["entity_id", "translation"])
    TRANS = dict(zip(t["entity_id"], t["translation"]))
    log(f"translations: {len(TRANS):,} records from {tr_files[0]}")
else:
    print("WARNING: translations_train.parquet not found: L8 runs without translations, L2t is skipped.")
mfiles = sorted(INPUT_ROOT.rglob("blocking_misses.csv")) + sorted(WORK.glob("blocking_misses.csv"))
assert mfiles, "blocking_misses.csv not found: attach it"
MISS = pd.read_csv(mfiles[0])
COUNTRIES = sorted(set(OLD) & set(NEWB))
recs = pd.concat([load_src(DATA_ROOT / "train" / f"train_source{i}.tsv") for i in (1, 2, 3)],
                 ignore_index=True).set_index("entity_id")
gt = pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 names=["s1", "m"], header=0, keep_default_na=False)
GT = {s: [x for x in m.split(",") if x] for s, m in zip(gt["s1"], gt["m"])}
del gt
log(f"countries {COUNTRIES}; {len(recs):,} records; {len(MISS):,} known misses")


def topk_rows(M, k):
    """Row-wise top-k of a CSR score matrix -> list of (cols, vals) per row."""
    out = []
    for i in range(M.shape[0]):
        s, e = M.indptr[i], M.indptr[i + 1]
        cols, vals = M.indices[s:e], M.data[s:e]
        if e - s > k:
            sel = np.argpartition(-vals, k)[:k]
            cols, vals = cols[sel], vals[sel]
        out.append((cols, vals))
    return out

In [ ]:
# %% [CELL 6] Per country: build L8 over the full pool, query, evaluate
RES, TIMING, FOUND_DEBUG = {}, {}, {}
for c in COUNTRIES:
    t_c = time.time()
    qids = np.load(OLD[c] / "qids.npy", allow_pickle=True)
    pids = np.load(OLD[c] / "pids.npy", allow_pickle=True)
    npool = len(pids)
    qpos = pd.Series(np.arange(len(qids)), index=qids)
    ppos = pd.Series(np.arange(npool), index=pids)
    # --- probe queries: held-out sample + S1 entities with known misses ---------
    rng = np.random.default_rng(SEED)
    qb = np.array([hash_bucket(x) for x in qids], dtype=np.int16)
    held = np.flatnonzero(qb < 500)
    held = rng.choice(held, min(N_HELDOUT, len(held)), replace=False)
    mc = MISS[MISS["country"] == c].copy()
    mc["q"] = qpos.reindex(mc["s1"]).to_numpy()
    mc["p"] = ppos.reindex(mc["cand"]).to_numpy()
    mc = mc.dropna(subset=["q", "p"]).astype({"q": np.int64, "p": np.int64})
    mq = mc["q"].unique()
    mq = rng.choice(mq, min(N_MISS_Q, len(mq)), replace=False)
    mc = mc[mc["q"].isin(mq)]
    probe_q = np.unique(np.concatenate([held, mq]))
    log(f"[{c}] pool {npool:,}; probe queries {len(probe_q):,} ({len(held):,} held-out, "
        f"{len(mq):,} with {len(mc):,} known misses)")

    # --- typed token matrices: pool (all) and probe queries ---------------------
    t0 = time.time()
    P_rec = recs.reindex(pids)
    Praw = build_tokens(P_rec["business_name"].tolist(), P_rec["business_address"].tolist(),
                        [TRANS.get(x, "") for x in pids], f"L8 pool {c}")
    Q_rec = recs.reindex(qids[probe_q])
    Qraw = build_tokens(Q_rec["business_name"].tolist(), Q_rec["business_address"].tolist(),
                        [""] * len(probe_q), f"L8 queries {c}")
    P, Q = {}, {}
    for t in TYPES:                                   # exact vocabulary per token type (pool)
        vocab = pd.Index(pd.unique(Praw[t][1]))
        P[t], Q[t] = to_csr(*Praw[t], vocab), to_csr(*Qraw[t], vocab)
        del vocab
    del Praw, Qraw
    t_tok = time.time() - t0
    # IDF per type; tokens in more than DF_CAP records get weight 0
    t0 = time.time()
    PT, sq_p, sq_q, Qs = {}, {}, {}, {}
    for t in TYPES:
        df = np.bincount(P[t].indices, minlength=P[t].shape[1])
        idf = np.where((df > 0) & (df <= DF_CAP), np.log(npool / np.maximum(df, 1)), 0.0).astype(np.float32)
        Ps = P[t].copy(); Ps.data = idf[Ps.indices]; Ps.eliminate_zeros()
        Qt = Q[t].copy(); Qt.data = idf[Qt.indices]; Qt.eliminate_zeros()
        sq_p[t] = np.asarray(Ps.multiply(Ps).sum(1)).ravel()     # per-record sum of idf^2
        sq_q[t] = np.asarray(Qt.multiply(Qt).sum(1)).ravel()
        PT[t] = Ps.T.tocsr()
        Qs[t] = Qt
        del Ps
    del P
    gc.collect()
    t_idx = time.time() - t0
    mem(f"{c} index built")

    # --- L8 search for every weight setting --------------------------------------
    found = {}
    t0 = time.time()
    # per-field norms (fixed type weights) and "has this field" flags
    fpn = {f: np.sqrt(sum(TYPE_W[t] ** 2 * sq_p[t] for t in ts)) for f, ts in FIELDS.items()}
    fqn = {f: np.sqrt(sum(TYPE_W[t] ** 2 * sq_q[t] for t in ts)) for f, ts in FIELDS.items()}
    pairs = {w: [] for w in WEIGHTS}
    for r0 in range(0, len(probe_q), Q_CHUNK):
        r1 = min(len(probe_q), r0 + Q_CHUNK)
        FM = {}
        for f, ts in FIELDS.items():                       # per-field cosine, sparse
            M = None
            for t in ts:
                part = (Qs[t][r0:r1] @ PT[t]) * (TYPE_W[t] ** 2)
                M = part if M is None else M + part
            M = M.tocsr()
            rows = np.repeat(np.arange(r1 - r0), np.diff(M.indptr))
            M.data /= np.maximum(fqn[f][r0:r1][rows] * fpn[f][M.indices], 1e-12)
            FM[f] = M
        for wname, FW in WEIGHTS.items():
            M = (FM["name"] * FW["name"] + FM["addr"] * FW["addr"]).tocsr()
            rows = np.repeat(np.arange(r1 - r0), np.diff(M.indptr))
            den = sum(FW[f] * (fqn[f][r0:r1][rows] > 0) * (fpn[f][M.indices] > 0) for f in FIELDS)
            M.data /= np.maximum(den, 1e-12)               # only fields BOTH records have
            for i, (cols, _) in enumerate(topk_rows(M, K8)):
                pairs[wname].append(probe_q[r0 + i] * npool + cols.astype(np.int64))
    for wname in WEIGHTS:
        found[wname] = np.unique(np.concatenate(pairs[wname])) if pairs[wname] else np.zeros(0, np.int64)
    t_search = time.time() - t0
    del PT, Qs
    gc.collect()

    # --- L2t: name+address TF-IDF over the non-Latin (translated) records only ----
    t0 = time.time()
    if HAS_GPU and TRANS:
        sub = np.flatnonzero(np.array([x in TRANS for x in pids]))
        if len(sub) > 50:
            ptxt = (clean_name(pd.Series([TRANS[x] for x in pids[sub]])) + " " +
                    clean_addr(P_rec["business_address"].iloc[sub].reset_index(drop=True))).tolist()
            qtxt = (clean_name(Q_rec["business_name"].reset_index(drop=True)) + " " +
                    clean_addr(Q_rec["business_address"].reset_index(drop=True))).tolist()
            vec, svd = fit_tfidf_svd(ptxt, DIM_L2T, (3, 4))
            Pv = tfidf_transform(vec, svd, ptxt, f"L2t pool {c}")
            Qv = tfidf_transform(vec, svd, qtxt, f"L2t queries {c}")
            I, S = gpu_topk(Qv, Pv, K2T, f"L2t search {c}")
            ok = I >= 0
            found["L2t"] = np.unique(probe_q[np.nonzero(ok)[0]] * npool + sub[I[ok]].astype(np.int64))
            del Pv, Qv
        log(f"[{c}] L2t over {len(sub):,} translated records")
    t_l2t = time.time() - t0

    # --- evaluation ---------------------------------------------------------------
    old = np.load(OLD[c] / "pairs.npz")
    nb = np.load(NEWB[c] / "pairs_b.npz")
    in_probe = np.zeros(len(qids), bool); in_probe[probe_q] = True
    have = np.union1d(old["q"][in_probe[old["q"]]].astype(np.int64) * npool + old["p"][in_probe[old["q"]]],
                      nb["q"][in_probe[nb["q"]]].astype(np.int64) * npool + nb["p"][in_probe[nb["q"]]])
    del old, nb
    # held-out true pairs
    tq, tp = [], []
    for q in held:
        for m in GT.get(qids[q], ()):
            j = ppos.get(m)
            if j is not None:
                tq.append(q); tp.append(j)
    tkeys = np.array(tq, dtype=np.int64) * npool + np.array(tp, dtype=np.int64)
    n_true = sum(len(GT.get(qids[q], ())) for q in held)
    mkeys = mc["q"].to_numpy() * npool + mc["p"].to_numpy()
    res = {"held_true": n_true, "held_found_old": int(np.isin(tkeys, have).sum()), "methods": {}}
    held_set = np.zeros(len(qids), bool); held_set[held] = True
    for name, keys in found.items():
        new_only = keys[~np.isin(keys, have)]
        res["methods"][name] = {
            "held_found": int(np.isin(tkeys, np.union1d(have, keys)).sum()),
            "new_per_query": float(held_set[new_only // npool].sum() / max(len(held), 1)),
            "miss_recovered": mc.assign(hit=np.isin(mkeys, keys)).groupby("category")["hit"].agg(["sum", "size"]),
        }
    if "L2t" in found:
        for wname in WEIGHTS:
            both = np.union1d(found[wname], found["L2t"])
            res["methods"][f"{wname} + L2t"] = {
                "held_found": int(np.isin(tkeys, np.union1d(have, both)).sum()),
                "new_per_query": float(held_set[both[~np.isin(both, have)] // npool].sum() / max(len(held), 1)),
                "miss_recovered": mc.assign(hit=np.isin(mkeys, both)).groupby("category")["hit"].agg(["sum", "size"]),
            }
    RES[c] = res
    TIMING[c] = dict(pool=npool, probe=len(probe_q), tokens=t_tok, index=t_idx,
                     search_per_query=t_search / max(len(probe_q), 1), l2t=t_l2t)
    FOUND_DEBUG[c] = {k: v.copy() for k, v in found.items()}      # kept for verification
    log(f"[{c}] done in {(time.time()-t_c)/60:.1f} min")
    del found, Q, P_rec, Q_rec
    gc.collect()

In [ ]:
# %% [CELL 7] Report
print("\n" + "=" * 92)
print("1. HELD-OUT RECALL  (random held-out entities; old blocking + Phase B, then + new layer)")
print("=" * 92)
for c, r in RES.items():
    base = r["held_found_old"] / max(r["held_true"], 1)
    print(f"  {c}: old + Phase B {base*100:6.2f}%   ({r['held_true']:,} true pairs)")
    for name, m in r["methods"].items():
        rec = m["held_found"] / max(r["held_true"], 1)
        print(f"     + {name:<26} {rec*100:6.2f}%  (+{(rec-base)*100:5.2f} pts)   "
              f"new candidates/query {m['new_per_query']:6.1f}")
print("\n" + "=" * 92)
print("2. KNOWN MISSES RECOVERED, by category  (share of the missed pairs of the probe queries)")
print("=" * 92)
for c, r in RES.items():
    names = list(r["methods"])
    cats = r["methods"][names[0]]["miss_recovered"].index.tolist()
    print(f"  {c}")
    print(f"    {'category':<40}{'misses':>8}" + "".join(f"{n[:18]:>20}" for n in names))
    for cat in cats:
        size = int(r["methods"][names[0]]["miss_recovered"].loc[cat, "size"])
        cells = "".join(f"{r['methods'][n]['miss_recovered'].loc[cat, 'sum']/max(size,1)*100:19.1f}%" for n in names)
        print(f"    {cat:<40}{size:>8,}{cells}")
    tot = sum(int(r["methods"][names[0]]["miss_recovered"]["size"].sum()) for _ in [0])
    cells = "".join(f"{r['methods'][n]['miss_recovered']['sum'].sum()/max(tot,1)*100:19.1f}%" for n in names)
    print(f"    {'ALL':<40}{tot:>8,}{cells}")
print("\n" + "=" * 92)
print("3. SPEED AND PROJECTED FULL-RUN TIME (L8 on CPU; L2t on GPU)")
print("=" * 92)
FULL_Q = {"test": 1_732_544, "train": 662_263}
for c, t in TIMING.items():
    print(f"  {c}: pool {t['pool']:,} records -> tokens {t['tokens']/60:.1f} min, index {t['index']/60:.1f} min; "
          f"search {t['search_per_query']*1000:.1f} ms/query; L2t {t['l2t']/60:.1f} min")
tok = sum(t["tokens"] + t["index"] for t in TIMING.values())
spq = np.mean([t["search_per_query"] for t in TIMING.values()])
for split, nq in FULL_Q.items():
    print(f"  projected full {split}: index ~{tok/60:.0f} min + search {nq:,} queries x {spq*1000:.1f} ms "
          f"= ~{(tok + nq*spq)/60:.0f} min (one weight setting; train pool; test pool is similar)")
log("L8 PROBE COMPLETE")